# step02b_suna_response_correction - the SUNA's response time, and whether correcting it fixes the bottle match

The SUNA reads nitrate in a flow cell, so it does not see the water it is in at once: it relaxes towards it with a
**response time** tau. On a profile this shows as **hysteresis**: going down the sensor still shows water from above,
coming up water from below, so the two legs disagree at the same depth. This stage measures tau from that
hysteresis, with **no bottles**, then asks the question that matters for the calibration: do lag-corrected profiles
match the bottles well enough to calibrate from depth-matched pairs?

**Model.** A first-order sensor obeys dy/dt = (x(t) - y(t)) / tau, where x(t) = N(z(t)) is the nitrate of the
water at the package's depth z(t) and N(z) is one profile, the same on both legs. For a given tau the reading is
linear in N, and integrating exactly between frames (x linear over each interval)

    y_i = a y_(i-1) + x_i (1 - r) + x_(i-1) (r - a),    a = exp(-dt / tau),   r = (tau / dt)(1 - a).

So for each tau the profile N is fitted by least squares on a 1 m grid (with a small curvature penalty), and the
residual sum of squares RSS(tau) says how well one profile seen through that lag explains both legs. tau is the
minimum of RSS over a grid. RSS(0) is the lag-free model ("both legs see the same profile"), so
1 - RSS(tau)/RSS(0) is the **share of the down/up mismatch the lag explains**. The model runs FORWARD (profile to
reading) and is fitted; it is not inverted. Inverting it (N = y + tau dy/dt) multiplies frame noise by about
tau/dt, which at tau = 30 s and 1 s frames swamps the signal.

**Statistics.** Frames within a cast are serially correlated, so each cast is one independent estimate of tau.
The cruise value is the median over casts, with the interquartile range and a bootstrap over casts (90%
interval of the median); a pooled value (RSS summed over casts) is shown too. Against the bottles, three
estimators are read at each bottle depth: the down-cast (1 m cells within +-2 m, step07's rule), the lag-free
fitted profile N@0, and the lag-corrected profile N@tau. Each is fitted as sensor = offset + gain x bottle
(OLS, 95% t interval on the gain), scored by leave-one-out cross-validation (LOOCV, bottle units), and compared
with the down-cast bottle by bottle (paired squared LOOCV errors, exact sign test).

**Clock first.** The SUNA logs on its own clock, set by hand before each cruise. If the cruise has a
`metadata/suna_time_offsets.csv` (`tools/set_suna_time_offset.py`, from step04's water-entry times), each frame's
depth is first corrected to the moment the frame was really taken: the co-located product is deliberately left
uncorrected (step04 measures the raw lag from it), so the correction is applied here. A clock offset is a pure
shift; left in, the first-order model absorbs it into tau (P45_07: 16 s of clock made tau read 25 to 29 s; with
the clock corrected tau is about 6 s on most casts).

**Decision, computed below, not written here.** The lag is "explained" if it removes at least `LAG_SHARE_MIN`
of the lag-free RSS over at least 3 casts. Lag-corrected depth-matched pairs are "usable for a calibration" only
if their R2 is at least `R2_USABLE`. This notebook never changes the calibration: the cruise's choice lives in
`metadata/suna_calibration.json` (tools/set_suna_calibration.py), and the verdict says whether the evidence
supports it.

**What changed, and why.** The earlier version of this stage inverted the lag and concluded, for P45_07, that "no
single tau reduces the down/up hysteresis; the mismatch only grows with tau ... near-total cell memory". That
verdict did not survive two corrections: (1) the inversion's noise amplification grows with tau, so its mismatch
grows with tau whatever the sensor does; (2) its time conversion, `pd.to_datetime(t).astype("int64") / 1e9`, is
seconds only for nanosecond storage, and pandas 3 stores these times in microseconds, so every frame interval came
out 1000 times too short and was clipped to the 0.1 s floor. With both removed (the forward model; seconds from
`suncat.timeutil`), P45_07 first gave tau of about 29 s, explaining about 92% of the hysteresis. Most of that
was the clock: the water-entry times put the SUNA 16.1 s late (95% CI 15.5 to 17.9 s, 27 casts, no drift over
244 h), and with the clock corrected tau is about 6 s on most casts, 28 to 47 s on four (01, 09, 10, 14; see
tools/diagnose_suna_lag.py). Against the bottles the clock plus a 6 s response gives R2 of about 0.70 and a
gain whose interval includes the stated 0.89; the single 29 s lag had given R2 0.56 and a gain of 1.18. The
homogeneous-layer calibration stays: depth-matched pairs in the nitracline remain too scattered (R2 below 0.90)
even with the timing and the response corrected. The model is shared with tools/estimate_response_time.py and
tools/diagnose_suna_lag.py (`suncat.nitrate.hysteresis`).

**Bottles are paired by one rule shared with tools/diagnose_suna_lag.py** (`suncat.nitrate.bottle_match`):
the same bottles (unflagged, step03's replicate QC), step07's down-cast cell median over the moving down-cast
frames, profiles read at the bottle depth inside each cast's grid, and a bottle compared only when every
estimator has a value (each one left out is counted with its reason). The two tools differ only in the frames
they fit: here the L2 co-located product, there step03's frames. Until Nov 2026 they used different bottle
lists and rules and compared 59 and 46 P45_07 bottles.

Sources: first-order sensor response and its correction, e.g. Bittig et al. (2014), *Limnol. Oceanogr.: Methods* 12, 617-636 (oxygen
optodes); Hastie, Tibshirani and Friedman, *The Elements of Statistical Learning*, ch. 7 (cross-validation);
Efron and Tibshirani (1993), *An Introduction to the Bootstrap*; UNESCO (1983) depth from pressure.

In [ ]:
import sys, json, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
try: get_ipython().run_line_magic("matplotlib", "inline")
except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image = None

try: _anchor = Path(__file__).resolve()
except NameError: _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists(): sys.path.insert(0, str(_p)); break
from project_paths import project_root, product_dir
try:
    import suncat.nitrate.hysteresis as H
except ImportError:                                   # suncat not installed: use the copy in the repo
    sys.path.insert(0, str(Path(_p) / "src"))
    import suncat.nitrate.hysteresis as H
from suncat.timeutil import elapsed_seconds
from suncat.calc.depth import pressure_to_depth
from suncat.suna_cal import load_suna_calibration
from suncat.suna_clock import load_offsets, describe as describe_offsets
import suncat.nitrate.bottle_match as BM          # the bottle pairing rule shared with tools/diagnose_suna_lag.py

CRUISE_ID  = "P45_07"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
L2_FILE    = product_dir(CRUISE_ID, "suna_l2") / f"{CRUISE_ID}__suna_L2_colocated.csv"
BOTTLE_NO3 = CRUISE_DIR / "metadata" / "bottle_nitrate" / f"{CRUISE_ID}_bottle_nitrate.csv"
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)

TAU_MAX_S     = 90.0    # longest tau tried (s); a cast whose minimum sits here is an EDGE, not an estimate
TAU_STEP_S    = 1.0     # tau grid step (s)
DZ_M          = 1.0     # profile grid (m)
SMOOTH        = H.SMOOTH
MIN_FRAMES    = H.MIN_FRAMES   # clean frames per leg for a cast to count
LAT_DEG       = 5.0     # latitude for depth from pressure (UNESCO 1983); 0 to 10 deg changes depth by < 0.01%
LAG_SHARE_MIN = 0.5     # the lag "explains" the hysteresis if it removes at least this share of the lag-free RSS
R2_USABLE     = 0.90    # lag-corrected depth-matched pairs are usable for a calibration only above this R2
N_BOOT        = 2000
SHOW          = None    # casts to plot; None = the three with the most bottles

_cal = load_suna_calibration(CRUISE_DIR)
print("cruise:", CRUISE_ID, "| stated calibration:", _cal["describe"])
CLOCK = load_offsets(CRUISE_DIR)                      # {cast: seconds added to SUNA time}; {} = none
print("clock:", describe_offsets(CRUISE_DIR))

L2 = pd.read_csv(L2_FILE, encoding="utf-8-sig")
L2["t_s"] = elapsed_seconds(L2["time_utc"])           # unit-safe (suncat.timeutil), any pandas version
if L2["t_s"].isna().any():
    print(f"  WARNING: {int(L2['t_s'].isna().sum())} frame times unreadable; those frames are dropped")
clean = L2["L1_clean"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])
inc = L2["in_cast"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])
use = L2[inc & L2["pressure_db"].notna() & L2["t_s"].notna()].copy()
use["z"] = pressure_to_depth(use["pressure_db"].to_numpy(float), LAT_DEG)
use["y"] = pd.to_numeric(use["nitrate_uM"], errors="coerce").where(clean.loc[use.index])
leg = use["leg"].astype(str).str.lower()
use["is_down"], use["is_up"] = leg.eq("down"), leg.eq("up")
CASTS = {}
for c, g in use.groupby("cast"):
    g = g.sort_values("t_s").copy()
    off = CLOCK.get(str(c), 0.0)
    if off:   # a frame stamped t was taken at t + off, when the package was at z(t + off): its true depth
        g["z"] = np.interp(g["t_s"] + off, g["t_s"], g["z"])
    g["moving"] = BM.moving(g["t_s"].to_numpy(float), g["z"].to_numpy(float))   # the down-cast estimator's frames
    CASTS[c] = g
print(f"{len(CASTS)} casts with in-cast frames; {int(use['y'].notna().sum())} clean frames with a value")

bot = BM.load_bottles(BOTTLE_NO3)                     # unflagged, finite, step03's replicate QC (suncat.nitrate.bottle_match)
print(f"{len(bot)} bottles after flag and replicate QC" + ("" if len(bot) else f" (no {BOTTLE_NO3.name}: the bottle test is skipped)"))


In [ ]:
# ------------------------------------------------------------------ tau per cast, no bottles
TAUS = np.arange(0.0, TAU_MAX_S + TAU_STEP_S / 2, TAU_STEP_S)
PER = {}
print(f"{'cast':16s} {'frames':>6s} {'down/up':>9s} {'tau (s)':>8s}   RMS misfit uM: tau=0 -> tau   up-minus-down: tau=0 -> tau")
for cid, g in CASTS.items():
    r = H.cast_tau(g["t_s"], g["z"], g["y"], g["is_down"], g["is_up"], TAUS, DZ_M, SMOOTH, MIN_FRAMES)
    PER[cid] = r
    if "skip" in r:
        print(f"{cid:16s} skipped: {r['skip']}", flush=True); continue
    ts = "  EDGE" if r["edge"] else f"{r['tau']:8.1f}"
    print(f"{cid:16s} {r['n']:6d} {r['n_down']:4d}/{r['n_up']:<4d} {ts:>8s}   {r['rms0']:7.3f} -> {r['rms']:6.3f}"
          f"           {r['hyst0']:+7.3f} -> {r['hyst']:+7.3f}", flush=True)

SUM = H.summarise(PER, TAUS, SMOOTH, N_BOOT)
n_edge = sum(bool(r.get("edge")) for r in PER.values())
if n_edge:
    print(f"\n{n_edge} cast(s) at the end of the grid (EDGE, no minimum below {TAU_MAX_S:g} s): left out; widen TAU_MAX_S if many")
if SUM:
    print(f"\ntau over {SUM['n_casts']} casts: median {SUM['tau_median']:.1f} s (IQR {SUM['tau_iqr'][0]:.1f} to {SUM['tau_iqr'][1]:.1f} s); "
          f"bootstrap over casts, 90% interval {SUM['tau_ci90'][0]:.1f} to {SUM['tau_ci90'][1]:.1f} s; pooled {SUM['tau_pooled']:.1f} s")
    print(f"the lag explains {SUM['rss_share_explained']:.0%} of the lag-free misfit (RSS summed over these casts)")
    print("tau is the e-folding time: 63% of a step change in tau seconds, 95% in 3 tau")
else:
    print("\nfewer than 3 casts with an estimate: no cruise value")

TAB = pd.DataFrame([{k: v for k, v in dict(cast=c, **r).items() if k not in ("curve", "data")} for c, r in PER.items()])
TAB.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_response_time.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------------ figure: RSS(tau)/RSS(0) per cast, and tau by cast
good = {c: r for c, r in PER.items() if np.isfinite(r.get("tau", np.nan))}
if good:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    for c, r in good.items():
        k = np.isfinite(r["curve"])
        a1.plot(TAUS[k], r["curve"][k] / r["rss0"], "-", lw=0.8, alpha=0.6)
    a1.set_xlabel("tau (s)"); a1.set_ylabel("RSS(tau) / RSS(0)"); a1.set_title("misfit against tau, per cast", fontsize=10)
    cs = list(good); tv = [good[c]["tau"] for c in cs]
    a2.plot(range(len(cs)), tv, "o", ms=4, color="#2E6FB0")
    if SUM:
        a2.axhline(SUM["tau_median"], color="#d62728", lw=1, label=f"median {SUM['tau_median']:.1f} s")
        a2.axhspan(*SUM["tau_ci90"], color="#d62728", alpha=0.12, label="90% interval (bootstrap over casts)")
        a2.legend(fontsize=8, frameon=False)
    a2.set_xticks(range(len(cs))); a2.set_xticklabels([c.split("_")[-1] for c in cs], fontsize=7)
    a2.set_xlabel("cast"); a2.set_ylabel("tau (s)"); a2.set_title("response time by cast", fontsize=10)
    for ax in (a1, a2):
        for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.tight_layout(); out = OUT_DIR / f"{CRUISE_ID}__suna_tau_by_cast.png"; fig.savefig(out, dpi=135)
    try: plt.close(fig)
    except Exception: pass
    if Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: pass
    print("saved", out.name)

In [ ]:
# ------------------------------------------------------------------ against the bottles (suncat.nitrate.bottle_match)
# One pairing rule with tools/diagnose_suna_lag.py: the same bottles, step07's down-cast cell median over MOVING
# down-cast frames (+-2 m), profiles read at the bottle depth inside each cast's grid, and a bottle compared only
# when every estimator has a value. What differs between the two tools is only the frames (printed below).
BOTTLES, FITS = pd.DataFrame(), {}
if len(bot) and SUM:
    tau_c = SUM["tau_median"]
    prof, frames = {}, {}
    for cid, r in good.items():
        t, z, y, grid = r["data"]
        prof[cid] = {"grid": grid, "N@0": H.fit_profile(t, z, y, grid, 0.0, SMOOTH)[1],
                     "N@tau": H.fit_profile(t, z, y, grid, tau_c, SMOOTH)[1]}
    for cid, g in CASTS.items():
        frames[cid] = dict(z=g["z"].to_numpy(float), y=g["y"].to_numpy(float),
                           down=(g["is_down"] & g["moving"]).to_numpy(bool))
    BOTTLES = BM.pair(bot, frames, prof, ["N@0", "N@tau"])
    CMP = BM.compare(BOTTLES, ["N@0", "N@tau"])
    print(f"tau = {tau_c:.1f} s (cruise median)")
    BM.report(CMP, "the L2 co-located product (SUNA frame times; step02's clock applied to the depths; legs as "
              "step02 marked them, down-cast moving frames)", "uM")
    FITS = CMP["fits"]
    BOTTLES.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_response_bottles.csv", index=False, encoding="utf-8-sig")
else:
    print("bottle test skipped (" + ("no bottles" if not len(bot) else "no cruise tau") + ")")


In [ ]:
# ------------------------------------------------------------------ verdict, from the numbers above
LAG_EXPLAINED = bool(SUM) and SUM["rss_share_explained"] >= LAG_SHARE_MIN
f = FITS.get("N@tau")
PAIRS_USABLE = bool(f) and f["R2"] >= R2_USABLE
print("VERDICT")
if not SUM:
    print(f"  no cruise tau (fewer than 3 casts with both legs): nothing to conclude")
elif LAG_EXPLAINED:
    print(f"  A first-order lag explains the down/up hysteresis: tau = {SUM['tau_median']:.1f} s (median over {SUM['n_casts']} casts; "
          f"90% interval {SUM['tau_ci90'][0]:.1f} to {SUM['tau_ci90'][1]:.1f} s) removes {SUM['rss_share_explained']:.0%} of the lag-free misfit.")
else:
    print(f"  A first-order lag does NOT explain the hysteresis: it removes only {SUM['rss_share_explained']:.0%} of the lag-free "
          f"misfit (< {LAG_SHARE_MIN:.0%}). The response is not first order, or the legs see different water.")
if f:
    gain_ok = _cal["mode"] == "adopted" and f["lo"] <= _cal["a1"] <= f["hi"]
    if PAIRS_USABLE:
        print(f"  Lag-corrected depth-matched pairs calibrate well (R2 {f['R2']:.3f} >= {R2_USABLE}; gain {f['gain']:.3f}, "
              f"95% CI {f['lo']:.3f} to {f['hi']:.3f}). A depth-matched calibration of the lag-corrected profiles is supported;")
        print("  this notebook does not change the stated calibration (tools/set_suna_calibration.py does).")
    else:
        print(f"  Lag-corrected depth-matched pairs do NOT calibrate well enough: R2 {f['R2']:.3f} < {R2_USABLE} "
              f"(down-cast {FITS['down']['R2']:.3f}); better than the down-cast on {f.get('better', 0)} of {f.get('n', 0)} bottles "
              f"(sign test p = {f.get('p_sign', float('nan')):.3f}).")
        print(f"  The scatter is not the lag. The stated calibration stays: {_cal['describe']}.")
    n0, dn = FITS.get("N@0"), FITS.get("down")
    if n0 and dn:
        # which estimator predicts the bottles best, and is any better than the down-cast? (stated from the numbers)
        lo = min((dn["loocv"], "down-cast"), (n0["loocv"], "lag-free fit of both legs"), (f["loocv"], "lag-corrected fit"))
        sig = [k for k, d in (("lag-free", n0), ("lag-corrected", f)) if d.get("p_sign", 1.0) < 0.05 and d.get("better", 0) > d.get("n", 0) / 2]
        print(f"  Against the bottles (LOOCV, uM): down-cast {dn['loocv']:.2f}, lag-free both legs {n0['loocv']:.2f}, "
              f"lag-corrected {f['loocv']:.2f}; lowest: {lo[1]}. "
              + (f"Significantly better than the down-cast: {', '.join(sig)}." if sig else
                 f"Neither both-leg fit is significantly better than the down-cast (sign tests p = "
                 f"{n0.get('p_sign', float('nan')):.3f} and {f.get('p_sign', float('nan')):.3f})."))
        if abs(f["loocv"] - n0["loocv"]) <= 0.1 * n0["loocv"]:
            print("  The lag correction itself changes the bottle match little (lag-corrected within 10% of lag-free).")
        _oo = globals().get("CMP", {}).get("one_out")
        if _oo:
            _rk = CMP["ranking"]
            print(f"  Robustness: the most influential bottle ({_oo['cast']} at {_oo['depth_m']:.1f} m) "
                  + ("does not change the ranking." if _rk["one_out"] == _rk["loocv"] else
                     "DECIDES the ranking: without it, " + " < ".join(_rk["one_out"]) + ". Check that bottle before "
                     "reading the ranking.")
                  + (f" By median |error| the order is {' < '.join(_rk['mae'])}." if _rk["mae"] != _rk["loocv"] else ""))
    if _cal["mode"] == "adopted":
        a1 = _cal["a1"]
        print(f"  The lag-corrected gain {f['gain']:.3f} (95% CI {f['lo']:.3f} to {f['hi']:.3f}) "
              + ("includes" if gain_ok else "EXCLUDES") + f" the stated gain {a1:g}.")
        if f["hi"] < a1:      # below: errors in the bottles pull an OLS gain down (regression dilution)
            print("  It lies BELOW it, the direction regression dilution (the bottles' own errors) pulls an OLS gain,")
            print("  so with this R2 the difference may be dilution, not the sensor.")
        elif f["lo"] > a1:    # above: dilution cannot do that
            print("  It lies ABOVE it. Regression dilution pulls gains DOWN, so it cannot explain this: in the nitracline")
            print("  the lag-corrected profile spans more nitrate than the bottles do. Candidates: the single-tau model")
            print("  over-corrects (tau varies between casts), or the bottles carry smoothed water (fired on the fly).")
summary = dict(cruise=CRUISE_ID, tau_max_s=TAU_MAX_S, clock=describe_offsets(CRUISE_DIR),
               clock_offsets_s=sorted(set(CLOCK.values())), lag_explained=LAG_EXPLAINED, pairs_usable=PAIRS_USABLE,
               lag_share_min=LAG_SHARE_MIN, r2_usable=R2_USABLE, calibration=_cal["describe"],
               **{k: v for k, v in SUM.items()},
               fits={k: {kk: (float(vv) if isinstance(vv, (int, float, np.floating, np.integer)) else vv) for kk, vv in d.items()}
                     for k, d in FITS.items()})
(OUT_DIR / f"{CRUISE_ID}__suna_response_summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("\nsaved", f"{CRUISE_ID}__suna_response_time.csv, __suna_response_bottles.csv, __suna_response_summary.json in", OUT_DIR)

In [ ]:
# ------------------------------------------------------------------ example casts: raw legs, lag-corrected profile, bottles
if SUM and good:
    counts = BOTTLES["cast"].value_counts() if len(BOTTLES) else pd.Series(dtype=int)
    show = SHOW or (list(counts.index[:3]) if len(counts) else list(good)[:3])
    for cid in show:
        if cid not in good: continue
        g = CASTS[cid]; t, z, y, grid = good[cid]["data"]
        Nt = H.fit_profile(t, z, y, grid, SUM["tau_median"], SMOOTH)[1]
        fig = plt.figure(figsize=(5.2, 6.4)); ax = fig.add_subplot(111)
        ax.plot(g.loc[g["is_down"], "y"], g.loc[g["is_down"], "z"], ".", ms=2, color="#9cbfe0", label="raw down")
        ax.plot(g.loc[g["is_up"], "y"], g.loc[g["is_up"], "z"], ".", ms=2, color="#f1b98e", label="raw up")
        ax.plot(Nt, grid, "-", lw=1.6, color="#222", label=f"lag-corrected, tau {SUM['tau_median']:.0f} s")
        bc = BOTTLES[BOTTLES["cast"] == cid] if len(BOTTLES) else pd.DataFrame()
        if len(bc) and _cal["mode"] == "adopted":      # bottles on the sensor's scale: a0 + a1 x bottle
            ax.plot(_cal["a0"] + _cal["a1"] * bc["bottle"], bc["depth_m"], "s", ms=7, color="#d62728",
                    label="bottle (on the SUNA scale, stated calibration)", zorder=5)
        ax.invert_yaxis(); ax.set_xlabel("NO3 (uM, raw SUNA scale)"); ax.set_ylabel("depth (m)")
        ax.set_title(cid, fontsize=10); ax.legend(fontsize=7, frameon=False)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
        fig.tight_layout(); out = OUT_DIR / f"{CRUISE_ID}__suna_respcorr__{cid}.png"; fig.savefig(out, dpi=130)
        try: plt.close(fig)
        except Exception: pass
        if Image is not None:
            try: display(Image(filename=str(out)))
            except Exception: pass
        print("saved", out.name)